## Indications
- This notebook recreates the graphical analysis on the forecast products, The content remain the same from the published paper
- Checkout the TODO flags to consider the reality of your file system structure
- You may need to install some module using the (%pip install MODULE_XYZ) in a separate code cell
- Please, if you yse this notebook for any official report or scientific activity, consider to cite the related paper and/or the code properly

In [ ]:
# %pip install xskillscore

In [ ]:
import os
import pandas as pd
import xskillscore as xs
import seaborn as sns
import xarray as xr
from pathlib import Path
from tqdm import tqdm
import pylab as plt
sns.set_style("ticks")

### Customize

In [ ]:
DATA_PAPER_PATH=r"Y:\repo_egu24\data_paper"     #TODO 1: Fix to your reality
CASE_="fr"                                      #TODO 2: Change between "us" and "fr" for yout runs
SAVE_IMG = True                                 #TODO 3: Turn True or False to manage image saving


idx={"us":0, "fr":1}.get(CASE_)
IMG_TO=rf"{DATA_PAPER_PATH}/images_user"  #TODO 4: Fix to your reality
os.makedirs(IMG_TO, exist_ok=True)

GENERAL = {"periods": [("19891001", "19910930"),("20180901", "20210831")],
           "years": [(1991, 2009),(1989,2018)],
           "ens_cases":[("climatology","hindcast"), ("climatology","hindcast","realtime")],
           "data_root": rf"{DATA_PAPER_PATH}/data/camels{CASE_}"}

COL_NAME = {'P_CM':"Rainfall",'SWI_CM_swe': "SWE", 'VP_CM': "Vp. Pressure",
            'T_CM_min':"T°.min", 'ETP_CM':"PET", 'T_CM_max': "T°.max", 
            'RAD_CM':"S. Rad.", "Q_Obs":"Discharge"}

DATA_ROOT = GENERAL["data_root"]
bv_list = [a.split()[0] for a in open(DATA_ROOT+"/basins_56", "r").readlines()]
PERIOD_ = GENERAL["periods"][idx]
YEARS_ = GENERAL["years"][idx]

In [ ]:
def get_pred_feature(f, feat, step:int=3):
    basin = Path(f).stem
    pr = pd.read_csv(f, sep=";", index_col=[0, 1, 2, 3], parse_dates=["time", "Date"])
    pr_var = pr[feat].xs(f"{step} days", level="step").droplevel("time").swaplevel().sort_index().unstack("member")
    pr_var["basin"] = f"{basin}".zfill(8)
    pr_var=pr_var.set_index("basin", append=True).swaplevel().sort_index()
    return pr_var

def get_obs_feature(f, feat):
    basin = Path(f).stem
    ob = pd.read_csv(f, sep=";", index_col="Date", parse_dates=["Date"])[feat].to_frame("obs")
    ob["basin"] = f"{basin}".zfill(8)
    ob=ob.set_index("basin", append=True).swaplevel().sort_index()
    return ob


def get_all_pred_feature(f,step:int=3):
    basin = Path(f).stem
    pr = pd.read_csv(f, sep=";", index_col=[0, 1, 2, 3], parse_dates=["time", "Date"])
    pr_var = pr.xs(f"{step} days", level="step").droplevel("time").swaplevel().sort_index()
    pr_var["basin"] = f"{basin}".zfill(8)
    pr_var=pr_var.set_index("basin", append=True).swaplevel().sort_index()
    return pr_var

def get_all_obs_feature(f):
    basin = Path(f).stem
    ob = pd.read_csv(f, sep=";", index_col="Date", parse_dates=["Date"])
    ob["basin"] = f"{basin}".zfill(8)
    ob=ob.set_index("basin", append=True).swaplevel().sort_index()
    return ob

In [ ]:
def make_rank_histogram(df_, bins_n=10):
    df= df_.copy()
    N_obs = df.groupby(level=df.index.names[:-1]).size().values[0]
    ds = df.to_xarray()

    # ens member
    members = [c for c in df.columns if "obs" not in str(c)]

    # Build ensemble
    ens = xr.concat([ds[m] for m in members], dim="member")
    ens = ens.assign_coords(member=members)  # dim(member, basin, Date)
    obs = ds["obs"]   # dim(basin, Date)
    
    # compute rank histogram
    rank = xs.rank_histogram(
        observations=obs,
        forecasts=ens,
        member_dim="member", dim=["Date"])
    ranks = rank.groupby_bins("rank", bins=bins_n).sum()
    ranks = ranks.assign_coords(rank_bins=range(1, bins_n + 1))
    ranks = ranks/N_obs
    return ranks

## RANK diagram for real time

In [ ]:
def make_multi_df_obs_forecast(forc_fold:str, step=1, root_data:str=DATA_ROOT, 
                                basin_list:list=bv_list, obs_fold="all_sim_obs_lstm"):
    """
    Group all forecast features together with their related observation

    :param
        - forc_fold: folder of forecast, like forc_fold/{basin_x.csv}
        - step: which time step to consider (1-7)
        - root_data: root directory for the expected data, like data_root/{forc_fold, obs_fold, basins_file}
        - basin_list: the list of basin to be considered, like [basin_i, basin_n]
        - obs_fold: the observed data folder name , like obs_fold/{basin_x.txt}
    """
    BV_BAR = tqdm(basin_list, desc=f"By-BV for {forc_fold} step {step} in {root_data}", leave=False)
    ALL_BV_RT = ()
    for b in BV_BAR:
        BV_BAR.set_postfix_str(f"BV: {b}")
        f_obs = f"{root_data}/{obs_fold}/{b}.txt"
        f_pr = f"{root_data}/{forc_fold}/{b}.csv"
        obs = get_all_obs_feature(f_obs)
        prd = get_all_pred_feature(f_pr,step=step)
        obs = (obs[prd.columns].melt(ignore_index=False, var_name="feat", value_name="obs")
                .set_index("feat", append=True)
                .reorder_levels(["feat", "basin", "Date"])
                .sort_index())
        prd = (prd.melt(ignore_index=False, var_name="feat", value_name="pred")
                .set_index("feat", append=True)
                .reorder_levels(["feat", "basin", "Date", "member"])
                .sort_index()
                ).pred.unstack("member").sort_index()
        both_op = pd.concat([obs, prd], axis=1, join="inner").sort_index()

        ALL_BV_RT+=(both_op, )
    ALL_BV_RT = pd.concat(ALL_BV_RT, axis=0).sort_index()
    return ALL_BV_RT

In [ ]:
def create_members(long_df, test_period=PERIOD_, clim_years=YEARS_,  base_year=("19890101", "19891231")):
    """Create forecast members based on a reference period, period, limit climatological year"""
    base_indx = pd.date_range(*base_year)
    ob_mbr = long_df.loc[long_df.index.get_level_values("Date").isin(pd.date_range(*test_period))]
    all_mbr = (ob_mbr,)
    for mb_x in range(*clim_years):
        i_y = mb_x-min(clim_years)
        new_ind = base_indx+pd.offsets.DateOffset(years=i_y)
        mbr_i = long_df.loc[long_df.index.get_level_values("Date").isin(new_ind)]["obs"].to_frame(i_y+1)
        mbr_i = ob_mbr.join(mbr_i.droplevel(["Date","Y"]), on=["feat","M_D"])
        all_mbr +=(mbr_i[[i_y+1]],)
    all_mbr = pd.concat(all_mbr, axis=1).droplevel(["Y", "M_D"]).sort_index()
    return all_mbr


In [ ]:
ls_df = []
for basin in tqdm(bv_list):
    df_x = pd.read_csv(DATA_ROOT+"/all_sim_obs_lstm" +  f"/{basin}.txt", sep=";", index_col="Date", parse_dates=True)
    L_df=df_x.melt(ignore_index=False, value_name="obs", var_name="feat").set_index("feat", append=True).swaplevel(0, 1).sort_index()
    L_df["M_D"] = L_df.index.get_level_values("Date").strftime("%m%d")
    L_df["Y"] = L_df.index.get_level_values("Date").strftime("%Y")
    L_df = L_df.set_index(["Y", "M_D"], append=True)
    L_df = create_members(L_df)
    L_df["basin"] = f'{basin}'.zfill(8)
    L_df=L_df.set_index("basin", append=True).swaplevel(-1, 1).sort_index()
    ls_df.append(L_df)
CL_DF=pd.concat(ls_df)
rk_cl = make_rank_histogram(CL_DF.rename(index=COL_NAME, level="feat"))

In [ ]:
fg, ax = plt.subplots(figsize=(6, 2), sharey=True)
r_kk = rk_cl.to_dataframe()
r_kk = r_kk.rename(index=COL_NAME, level="feat")
sns.barplot(r_kk, hue="feat", x="rank_bins", y="histogram_rank",linewidth=0.5,
            hue_order=list(COL_NAME.values()),
            ax=ax, capsize=0.4, palette="rainbow", width=0.8,
            err_kws={"lw":0.7, "color":"k"})
ax.axhline(1/len(r_kk.index.get_level_values("rank_bins").unique()), color="r", ls="--", lw=0.9)
ax.set(ylabel="Frequency", xlabel="Rank bins", title=f"Climatology-{CASE_}", ylim=(0, .3))
ax.grid(axis="y", ls="--", lw=0.3, color="k")
ax.legend(ncols=4, bbox_to_anchor=(.95, -.3), 
          columnspacing=0.3)
if SAVE_IMG is True:
    plt.savefig(rf"{IMG_TO}\clim_rankD_{CASE_}_all_feat.png", bbox_inches="tight", dpi=300)

In [ ]:
STEP_=3
HIND_FORC_ = make_multi_df_obs_forecast("hindcast", step=STEP_)
HIND_FORC_1 = HIND_FORC_.rename(index=COL_NAME, level="feat")
rk_hd = make_rank_histogram(HIND_FORC_1)
rk_rl = None
if CASE_ == "fr":
    REAL_FORC_ = make_multi_df_obs_forecast("realtime", step=STEP_)
    REAL_FORC_1 = REAL_FORC_.rename(index=COL_NAME, level="feat")
    rk_rl = make_rank_histogram(REAL_FORC_1)

In [ ]:
fg, axs = plt.subplots(figsize=(6, 2)) if "us" in CASE_ else plt.subplots(1,2,figsize=(11, 2.2), sharey=True)
for i, r_k_ in enumerate([rk_hd, rk_rl][:(1 if "us" in CASE_ else None)]):
    ax=axs if "us" in CASE_ else axs[i]
    r_kk = r_k_.to_dataframe()
    sns.barplot(r_kk, hue="feat", x="rank_bins", y="histogram_rank",linewidth=0.5,
                hue_order=list(COL_NAME.values())[:-1],
                ax=ax, capsize=0.4, palette="rainbow", width=0.8,
                err_kws={"lw":0.7, "color":"k"})
    ax.axhline(1/len(r_kk.index.get_level_values("rank_bins").unique()), color="k", ls="--", lw=0.9)
    ax.set(ylabel="Frequency", xlabel="Rank bins", title=["Hindcast","Forecast Archives",][i]+f"-{CASE_.upper()}", ylim=(0, 0.8))
    ax.grid(axis="y", ls="--", lw=0.3, color="k")
    ax.legend(ncols=4, bbox_to_anchor=(.92 if "us" in CASE_ else 1., -.3), 
              columnspacing=0.3)
plt.subplots_adjust(wspace=0.1)
if SAVE_IMG is True:
    plt.savefig(rf"{IMG_TO}\forecast_products_rankD_{CASE_}_hp{STEP_}.png", bbox_inches="tight", dpi=300)

## CLIM against Hindcast

In [ ]:
SHOW_FEAT = ["P_CM", "ETP_CM"]
SHOW_F = [COL_NAME[a] for a in SHOW_FEAT]
fg, axs = plt.subplots(1,2,figsize=(10, 2.5), sharey=True)
for i, r_k_ in enumerate([rk_cl, rk_hd]):
    ax=axs[i]
    r_kk = r_k_.to_dataframe()
    r_kk = r_kk[r_kk.index.get_level_values("feat").isin(SHOW_F)]
    sns.barplot(r_kk, hue="feat", x="rank_bins", 
                y="histogram_rank",linewidth=0.5,
                hue_order=SHOW_F,
                ax=ax, capsize=0.4, palette=["teal", "silver"], width=0.8,
                err_kws={"lw":0.7, "color":"k"})
    ax.axhline(1/len(r_kk.index.get_level_values("rank_bins").unique()), color="r", ls="-", lw=0.9)
    ax.set(ylabel="Frequency", xlabel="Rank bins", 
           title=["Climatology","Hindcast",][i]+f'-{CASE_.upper()}', 
           ylim=(0, .42 if "us" in CASE_ else 0.5))
    ax.grid(axis="y", ls="--", lw=0.3, color="k")
    if i==1:
        ax.legend(ncols=8, bbox_to_anchor=(.2, -.2), columnspacing=0.5)
    else:
        ax.get_legend().remove()
plt.subplots_adjust(wspace=0.1)
if SAVE_IMG is True:
    plt.savefig(rf"{IMG_TO}\Clim_vs_Hindcast_rankD_P_PET_{CASE_}_hp{STEP_}.png", bbox_inches="tight", dpi=300)

## HINDCAST vs Lead time

In [ ]:
for FCST_PROD in ["hindcast", "realtime"][:(1 if "us" in CASE_ else None)]:
    SHOW_FEAT = ["P_CM", "ETP_CM"]
    SHOW_F = [COL_NAME[a] for a in SHOW_FEAT]
    fig_hp, axs=plt.subplots(1,3, figsize=(9, 2.), sharey="all")
    for i, step_hp in enumerate([1, 3, 7]):
        HIND_FORC_hp = make_multi_df_obs_forecast(FCST_PROD, step=step_hp)
        rk_hd_hp = make_rank_histogram(HIND_FORC_hp.rename(index=COL_NAME, level="feat"))
        ax=axs[i]
        r_kk = rk_hd_hp.to_dataframe()
        r_kk = r_kk[r_kk.index.get_level_values("feat").isin(SHOW_F)]
        sns.barplot(r_kk, hue="feat", x="rank_bins", 
                    y="histogram_rank",
                    linewidth=0.5,
                    hue_order=SHOW_F,
                    ax=ax, capsize=0.4, palette=["teal", "silver"], width=0.8,
                    err_kws={"lw":0.7, "color":"k"})
        ax.axhline(1/len(r_kk.index.get_level_values("rank_bins").unique()), color="r", ls="-", lw=0.9)
        ax.set(ylabel="Frequency", xlabel="Rank bins", 
               title=FCST_PROD.title()+f'-{CASE_.upper()} (hp={step_hp} day{"s" if step_hp>1 else ""})', #Corrected from '1 day' to 'step_hp day' on May24 2026
               ylim=(0, .7 if "us" in CASE_ else 0.6))
        ax.grid(axis="y", ls="--", lw=0.3, color="k")
        ax.legend(loc="upper center")
    plt.subplots_adjust(wspace=0.1)
    if SAVE_IMG is True:
        plt.savefig(rf"{IMG_TO}\{FCST_PROD.title()}_rankD_P_PET_{CASE_}_hp1_3_7_c_{FCST_PROD}.png", bbox_inches="tight", dpi=300)

---
---